# Notebook 5 — Cuadro completo de eliminatoria (bracket) con predicción en cascada

**Objetivo**: construye UN único cuadro determinista, partido a partido, de dieciseisavos a la
final, usando:
- El resultado **real** para cualquier cruce que ya se haya jugado.
- La predicción del **modelo más reciente** (el checkpoint más nuevo que dejó el Notebook 4,
  sea cual sea su familia) para cualquier cruce con ambos equipos ya conocidos pero sin jugar
  todavía.
- Esa misma predicción **en cascada** para resolver las rondas siguientes: el ganador previsto
  de un cruce alimenta al cruce del que depende, hasta llegar a un campeón.

A diferencia de la simulación Montecarlo del Notebook 4 (miles de sorteos, probabilidades por
ronda), aquí hay una única predicción por partido -- el "camino más probable" completo, bueno
para visualizar pero sin cuantificar la incertidumbre de cada cruce.

Salida: `results/cuadro_completo.csv` (una fila por partido, real o previsto) y
`results/bracket.html` (visualización con banderas, autocontenida).

In [1]:
import json
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.stats import poisson

DIR_RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_PROCESSED = DIR_RAIZ / "data" / "processed"
DIR_RAW = DIR_RAIZ / "data" / "raw"
DIR_RESULTS = DIR_RAIZ / "results"
DIR_RESULTS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)

MAX_GOLES_MATRIZ = 8
FECHA_CORTE_HISTORICO = pd.Timestamp("1990-01-01")
# FEATURES_MODELO se fija más abajo, leída del metadata.json del checkpoint
# (no se hardcodea: evita que este notebook se desincronice del Notebook 2
# cuando se añaden features nuevas -- justo lo que pasó antes de este cambio).

# Checkpoint más reciente del Notebook 4 (el de mayor `ultima_fecha_entrenamiento`,
# excluyendo "pre_mundial" -- por definición el más antiguo): sea cual sea la
# etapa del torneo o la familia ganadora, siempre el "estado del arte" de hoy,
# sin necesidad de actualizar a mano un nombre de carpeta.
DIR_CHECKPOINTS = DIR_RAIZ / "models" / "checkpoints"


def checkpoint_mas_reciente(dir_checkpoints: Path) -> str:
    candidatos = [
        (json.loads((carpeta / "metadata.json").read_text())["ultima_fecha_entrenamiento"], carpeta.name)
        for carpeta in dir_checkpoints.iterdir() if carpeta.is_dir() and carpeta.name != "pre_mundial"
    ]
    return max(candidatos)[1] if candidatos else "pre_mundial"


RUTA_CHECKPOINT = DIR_CHECKPOINTS / checkpoint_mas_reciente(DIR_CHECKPOINTS)
metadata_checkpoint = json.loads((RUTA_CHECKPOINT / "metadata.json").read_text())
FEATURES_MODELO = metadata_checkpoint["features"]
# Agnostico a la familia -- solo se llama a `.predict()` sobre lo que sea que
# el Notebook 4 haya guardado como ganador (GLM, LightGBM o XGBoost).
modelo_local = joblib.load(RUTA_CHECKPOINT / "modelo_goles_local.joblib")
modelo_visitante = joblib.load(RUTA_CHECKPOINT / "modelo_goles_visitante.joblib")
etapa = metadata_checkpoint["etapa"]
familia_modelo = metadata_checkpoint["familia"]
ultima_fecha = metadata_checkpoint["ultima_fecha_entrenamiento"]
n_partidos = metadata_checkpoint["n_partidos_entrenamiento"]
print(f"Checkpoint cargado: '{etapa}', entrenado hasta {ultima_fecha} ({n_partidos:,} partidos).")

df = pd.read_csv(DIR_PROCESSED / "partidos_features.csv", parse_dates=["fecha"])
df = df[df["fecha"] >= FECHA_CORTE_HISTORICO].reset_index(drop=True)
df_jugados = df[df["jugado"]].copy()
print(f"Partidos jugados desde {FECHA_CORTE_HISTORICO.date()}: {len(df_jugados):,}")

Checkpoint cargado: 'post_ronda_1', entrenado hasta 2026-07-04 (32,375 partidos).


Partidos jugados desde 1990-01-01: 32,375


## 5.1 Corrección de Dixon-Coles

Igual que en los Notebook 4: se recalibra `rho` sobre todo el histórico jugado
disponible (este notebook se ejecuta de forma independiente, así que no hereda el `rho`
de otro notebook).

In [2]:
def log_verosimilitud_dixon_coles(rho: float, lam: np.ndarray, mu: np.ndarray, x: np.ndarray, y: np.ndarray) -> float:
    tau = np.ones_like(lam)
    es_0_0 = (x == 0) & (y == 0)
    es_0_1 = (x == 0) & (y == 1)
    es_1_0 = (x == 1) & (y == 0)
    es_1_1 = (x == 1) & (y == 1)
    tau[es_0_0] = 1 - lam[es_0_0] * mu[es_0_0] * rho
    tau[es_0_1] = 1 + lam[es_0_1] * rho
    tau[es_1_0] = 1 + mu[es_1_0] * rho
    tau[es_1_1] = 1 - rho
    tau = np.clip(tau, 1e-8, None)
    return float((np.log(tau) + poisson.logpmf(x, lam) + poisson.logpmf(y, mu)).sum())


X_jugados = df_jugados[FEATURES_MODELO]
lam_hist = np.clip(modelo_local.predict(X_jugados), 0.01, None)
mu_hist = np.clip(modelo_visitante.predict(X_jugados), 0.01, None)
x_hist, y_hist = df_jugados["goles_local"].to_numpy(), df_jugados["goles_visitante"].to_numpy()

candidatos_rho = np.linspace(-0.3, 0.3, 121)
log_verosimilitudes = [log_verosimilitud_dixon_coles(r, lam_hist, mu_hist, x_hist, y_hist) for r in candidatos_rho]
RHO_DIXON_COLES = float(candidatos_rho[np.argmax(log_verosimilitudes)])
print(f"rho = {RHO_DIXON_COLES:.4f}")


def matriz_probabilidad_conjunta(lam_a: float, lam_b: float, max_goles: int = MAX_GOLES_MATRIZ) -> np.ndarray:
    goles = np.arange(max_goles + 1)
    matriz = np.outer(poisson.pmf(goles, lam_a), poisson.pmf(goles, lam_b))
    matriz[0, 0] *= 1 - lam_a * lam_b * RHO_DIXON_COLES
    matriz[0, 1] *= 1 + lam_a * RHO_DIXON_COLES
    matriz[1, 0] *= 1 + lam_b * RHO_DIXON_COLES
    matriz[1, 1] *= 1 - RHO_DIXON_COLES
    matriz = np.clip(matriz, 0, None)
    return matriz / matriz.sum()


def resolver_cruce(equipo_a: str, equipo_b: str, lam_a: float, lam_b: float) -> dict:
    """Misma regla que `resolver_eliminatoria` del Notebook 4: quién avanza lo
    decide SIEMPRE la probabilidad acumulada en la matriz conjunta, nunca la
    comparación puntual de los dos lambdas. `marcador_previsto` es el argmax
    de esa misma matriz -- no floor(lambda) de cada lado por separado, que
    puede diferir del argmax conjunto una vez Dixon-Coles introduce
    correlación entre los dos lados (ver Notebook 4)."""
    matriz = matriz_probabilidad_conjunta(lam_a, lam_b)
    prob_a = np.tril(matriz, -1).sum()
    prob_b = np.triu(matriz, 1).sum()
    ganador = equipo_a if prob_a > prob_b else equipo_b
    fila_moda, col_moda = np.unravel_index(np.argmax(matriz), matriz.shape)
    return {"marcador_previsto": f"{fila_moda}-{col_moda}",
            "ganador": ganador, "prob_a": float(prob_a), "prob_b": float(prob_b)}

rho = -0.0500


## 5.2 El cuadro como grafo de dependencias, con sede de cada partido

Mismo enfoque que el Notebook 6 (6.2): los cruces de octavos en adelante se normalizan a
referencias puras `W<num>`/`L<num>` buscando en qué partido anterior quedó cada selección
como ganadora o perdedora REAL -- el calendario JSON a veces no ha sustituido todavía un
placeholder por el nombre real aunque el partido que lo decide ya se jugó (p.ej. el
partido 93 sigue listando "W83"/"W84" aunque 83 y 84 ya tienen marcador), así que no basta
con fiarse de `team1`/`team2` tal cual.

También se guarda el país anfitrión de la sede de cada partido (las 16 sedes están fijadas
de antemano para todo el torneo, incluida la final) para poder calcular `es_neutral` en
cualquier cruce, jugado o hipotético.

In [3]:
RUTA_CALENDARIO_JSON = DIR_RAW / "wc2026_calendario.json"
MAPEO_NOMBRES_JSON = {
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "USA": "United States",
}
CIUDAD_A_PAIS_MUNDIAL_2026 = {
    "Mexico City": "Mexico", "Zapopan": "Mexico", "Guadalupe": "Mexico",
    "Toronto": "Canada", "Vancouver": "Canada",
    "Inglewood": "United States", "Santa Clara": "United States",
    "East Rutherford": "United States", "Foxborough": "United States",
    "Philadelphia": "United States", "Houston": "United States",
    "Seattle": "United States", "Atlanta": "United States",
    "Miami Gardens": "United States", "Arlington": "United States",
    "Kansas City": "United States",
}
ROUND_POR_NUM = {
    **{n: "Dieciseisavos de final" for n in range(73, 89)},
    **{n: "Octavos de final" for n in range(89, 97)},
    **{n: "Cuartos de final" for n in range(97, 101)},
    101: "Semifinales", 102: "Semifinales",
    103: "Tercer puesto", 104: "Final",
}


def _ciudad_desde_sede(sede: str) -> str:
    return sede.split("(")[1].rstrip(")") if "(" in sede else sede


with open(RUTA_CALENDARIO_JSON, encoding="utf-8") as f:
    calendario = json.load(f)["matches"]
partidos_json = {m["num"]: m for m in calendario if "num" in m}


def ganador_real(partido: dict) -> str | None:
    marcador = partido.get("score")
    if marcador is None:
        return None
    for clave in ("p", "et", "ft"):
        if clave in marcador:
            g1, g2 = marcador[clave]
            if g1 != g2:
                return partido["team1"] if g1 > g2 else partido["team2"]
    return None


def marcador_real_str(partido: dict) -> str | None:
    marcador = partido.get("score")
    if marcador is None or "ft" not in marcador:
        return None
    g1, g2 = marcador["ft"]
    return f"{g1}-{g2}"


equipo_a_num = {}  # selección -> nº de partido (73-88) del que salió ganadora real
for num, partido in partidos_json.items():
    if num > 88:
        continue
    ganador = ganador_real(partido)
    if ganador:
        equipo_a_num[MAPEO_NOMBRES_JSON.get(ganador, ganador)] = num


def normalizar_referencia(referencia: str) -> str:
    equipo = MAPEO_NOMBRES_JSON.get(referencia, referencia)
    return f"W{equipo_a_num[equipo]}" if equipo in equipo_a_num else equipo


bracket = {}
for num, partido in sorted(partidos_json.items()):
    if num <= 88:
        referencia_a = MAPEO_NOMBRES_JSON.get(partido["team1"], partido["team1"])
        referencia_b = MAPEO_NOMBRES_JSON.get(partido["team2"], partido["team2"])
    else:
        referencia_a = normalizar_referencia(partido["team1"])
        referencia_b = normalizar_referencia(partido["team2"])
    ciudad = _ciudad_desde_sede(partido["ground"])
    bracket[num] = {
        "ronda": ROUND_POR_NUM[num],
        "fecha": pd.Timestamp(partido["date"]),
        "referencia_a": referencia_a, "referencia_b": referencia_b,
        "pais_sede": CIUDAD_A_PAIS_MUNDIAL_2026[ciudad],
        "ganador_real": ganador_real(partido),
        "marcador_real": marcador_real_str(partido),
    }

bracket_ordenado = sorted(bracket.items())
print(f"{len(bracket_ordenado)} partidos en el cuadro, de dieciseisavos a la final.")

32 partidos en el cuadro, de dieciseisavos a la final.


## 5.3 Snapshot actual y H2H actual por selección

A diferencia del Notebook 6 (snapshot *congelado* antes del Mundial), aquí interesa el
estado **más reciente conocido** de cada selección -- su última fila jugada, sea de antes
o de dentro del Mundial 2026 -- porque este cuadro parte de HOY, no del día 0 del torneo.
Mismo motivo para el historial cara a cara: se recalcula sobre TODO lo jugado hasta ahora,
que ya puede incluir algún cruce de este mismo Mundial.

In [4]:
def snapshot_actual(equipo: str, df_jugados: pd.DataFrame) -> dict:
    es_local = df_jugados["equipo_local"] == equipo
    es_visitante = df_jugados["equipo_visitante"] == equipo
    ultima_local = df_jugados.loc[es_local, "fecha"].max() if es_local.any() else pd.Timestamp.min
    ultima_visitante = df_jugados.loc[es_visitante, "fecha"].max() if es_visitante.any() else pd.Timestamp.min

    if ultima_local >= ultima_visitante:
        fila = df_jugados.loc[es_local].sort_values("fecha").iloc[-1]
        sufijo = "_local"
    else:
        fila = df_jugados.loc[es_visitante].sort_values("fecha").iloc[-1]
        sufijo = "_visitante"

    return {
        "elo": fila[f"elo_actual{sufijo}"],
        "tendencia_elo": fila[f"tendencia_elo{sufijo}"],
        "forma_gf_5": fila[f"prom_gf_5{sufijo}"],
        "forma_gf_10": fila[f"prom_gf_10{sufijo}"],
        "racha_5": fila[f"racha_puntos_5{sufijo}"],
        "racha_10": fila[f"racha_puntos_10{sufijo}"],
        "titulos_8a": fila[f"titulos_8a{sufijo}"], "titulos_4a": fila[f"titulos_4a{sufijo}"],
        "finales_8a": fila[f"finales_8a{sufijo}"], "ppp_vs_top_3a": fila[f"ppp_vs_top_3a{sufijo}"],
        "valor_plantilla": fila[f"valor_plantilla{sufijo}"],
        "ultima_fecha": max(ultima_local, ultima_visitante),
    }


def calcular_h2h_actual(equipos: list[str], df_jugados: pd.DataFrame) -> dict[tuple[str, str], tuple[float, float]]:
    """(puntos_prom, dif_goles_prom) de x contra y, sobre TODO lo jugado hasta
    ahora entre esa pareja -- ver `calcular_h2h` del Notebook 2 para la misma
    definición aplicada partido a partido en vez de como snapshot único."""
    resultados: dict[tuple[str, str], tuple[float, float]] = {}
    for i, x in enumerate(equipos):
        for y in equipos[i + 1:]:
            previos = df_jugados[((df_jugados["equipo_local"] == x) & (df_jugados["equipo_visitante"] == y))
                                  | ((df_jugados["equipo_local"] == y) & (df_jugados["equipo_visitante"] == x))]
            if previos.empty:
                resultados[(x, y)] = (1.0, 0.0)
                resultados[(y, x)] = (1.0, 0.0)
                continue
            puntos_x, puntos_y, difs_x, difs_y = [], [], [], []
            for fila in previos.itertuples(index=False):
                if fila.equipo_local == x:
                    gx, gy = fila.goles_local, fila.goles_visitante
                else:
                    gx, gy = fila.goles_visitante, fila.goles_local
                puntos_x.append(3 if gx > gy else (1 if gx == gy else 0))
                puntos_y.append(3 if gy > gx else (1 if gx == gy else 0))
                difs_x.append(gx - gy)
                difs_y.append(gy - gx)
            resultados[(x, y)] = (float(np.mean(puntos_x)), float(np.mean(difs_x)))
            resultados[(y, x)] = (float(np.mean(puntos_y)), float(np.mean(difs_y)))
    return resultados


equipos_dieciseisavos = sorted({p["referencia_a"] for _, p in bracket_ordenado if not re.match(r"^[WL]\d+$", p["referencia_a"])}
                                 | {p["referencia_b"] for _, p in bracket_ordenado if not re.match(r"^[WL]\d+$", p["referencia_b"])})

snapshots = {equipo: snapshot_actual(equipo, df_jugados) for equipo in equipos_dieciseisavos}
h2h_actual = calcular_h2h_actual(equipos_dieciseisavos, df_jugados)
print(f"Snapshot y H2H actuales calculados para {len(snapshots)} selecciones.")

Snapshot y H2H actuales calculados para 32 selecciones.


## 5.4 Cascada determinista: de dieciseisavos a la final

Para cada partido del cuadro, en orden de número (toda dependencia `W<num>`/`L<num>`
apunta a un número menor, así que un único paso por `bracket_ordenado` basta):

1. Resolver quién juega, tirando de `ganadores`/`perdedores` si hace falta.
2. Si el partido YA se jugó de verdad (`bracket[num]["ganador_real"]`), usar ese
   resultado tal cual -- no tiene sentido "predecir" un partido ya jugado.
3. Si no, predecir con el modelo `post_ronda_1` + snapshot/H2H actuales, y decidir el
   ganador con la matriz conjunta (7.1), igual que en el resto del proyecto.

El equipo anfitrión de la sede (si juega) va siempre en el slot "local" de las features
-- misma convención que el resto del pipeline (Notebook 1, `_determinar_local_visitante`).

In [5]:
def construir_features(equipo_a: str, equipo_b: str, fecha: pd.Timestamp, es_neutral: bool,
                        ultimo_partido: dict) -> pd.DataFrame:
    snap_a, snap_b = snapshots[equipo_a], snapshots[equipo_b]
    h2h_puntos, h2h_dif_goles = h2h_actual[(equipo_a, equipo_b)]
    return pd.DataFrame([{
        "elo_diff": snap_a["elo"] - snap_b["elo"],
        "tendencia_elo_local": snap_a["tendencia_elo"],
        "tendencia_elo_visitante": snap_b["tendencia_elo"],
        "dif_forma_gf_5": snap_a["forma_gf_5"] - snap_b["forma_gf_5"],
        "dif_forma_gf_10": snap_a["forma_gf_10"] - snap_b["forma_gf_10"],
        "dif_racha_5": snap_a["racha_5"] - snap_b["racha_5"],
        "dif_racha_10": snap_a["racha_10"] - snap_b["racha_10"],
        "dias_descanso_local": (fecha - ultimo_partido[equipo_a]).days,
        "dias_descanso_visitante": (fecha - ultimo_partido[equipo_b]).days,
        "es_neutral": es_neutral,
        "h2h_puntos_prom": h2h_puntos,
        "h2h_dif_goles_prom": h2h_dif_goles,
        "dif_titulos_8a": snap_a["titulos_8a"] - snap_b["titulos_8a"],
        "dif_titulos_4a": snap_a["titulos_4a"] - snap_b["titulos_4a"],
        "dif_finales_8a": snap_a["finales_8a"] - snap_b["finales_8a"],
        "dif_ppp_vs_top_3a": snap_a["ppp_vs_top_3a"] - snap_b["ppp_vs_top_3a"],
        "dif_valor_plantilla": snap_a["valor_plantilla"] - snap_b["valor_plantilla"],
    }])[FEATURES_MODELO]


def resolver_equipo(referencia: str, ganadores: dict, perdedores: dict) -> str:
    m = re.match(r"^([WL])(\d+)$", referencia)
    if m is None:
        return referencia
    tipo, num = m.group(1), int(m.group(2))
    return ganadores[num] if tipo == "W" else perdedores[num]


ultimo_partido = {equipo: snap["ultima_fecha"] for equipo, snap in snapshots.items()}
ganadores, perdedores = {}, {}
filas_cuadro = []

for num, partido in bracket_ordenado:
    equipo_a = resolver_equipo(partido["referencia_a"], ganadores, perdedores)
    equipo_b = resolver_equipo(partido["referencia_b"], ganadores, perdedores)
    fecha = partido["fecha"]
    pais_sede = partido["pais_sede"]

    if equipo_b == pais_sede and equipo_a != pais_sede:
        equipo_a, equipo_b = equipo_b, equipo_a
    es_neutral = equipo_a != pais_sede

    if partido["ganador_real"] is not None:
        ganador_json = MAPEO_NOMBRES_JSON.get(partido["ganador_real"], partido["ganador_real"])
        marcador = partido["marcador_real"]
        ganador = ganador_json
        jugado = True
        prob_gana_a = prob_gana_b = None
    else:
        X = construir_features(equipo_a, equipo_b, fecha, es_neutral, ultimo_partido)
        lam_a = float(np.clip(modelo_local.predict(X), 0.01, None)[0])
        lam_b = float(np.clip(modelo_visitante.predict(X), 0.01, None)[0])
        resultado = resolver_cruce(equipo_a, equipo_b, lam_a, lam_b)
        marcador = resultado["marcador_previsto"]
        ganador = resultado["ganador"]
        jugado = False
        prob_gana_a, prob_gana_b = resultado["prob_a"], resultado["prob_b"]

    perdedores[num] = equipo_b if ganador == equipo_a else equipo_a
    ganadores[num] = ganador
    ultimo_partido[equipo_a] = fecha
    ultimo_partido[equipo_b] = fecha

    filas_cuadro.append({
        "num_partido": num, "ronda": partido["ronda"], "fecha": fecha,
        "equipo_a": equipo_a, "equipo_b": equipo_b, "es_neutral": es_neutral,
        "marcador": marcador, "ganador": ganador, "jugado": jugado,
        "prob_gana_a": prob_gana_a, "prob_gana_b": prob_gana_b,
    })
    estado = "REAL" if jugado else "PREVISTO"
    print(f"  [{estado}] {partido['ronda']} ({num}): {equipo_a} {marcador} {equipo_b} -> avanza {ganador}")

df_cuadro = pd.DataFrame(filas_cuadro)
print(f"\nCampeón previsto: {ganadores[104]}")

  [REAL] Dieciseisavos de final (73): South Africa 0-1 Canada -> avanza Canada
  [REAL] Dieciseisavos de final (74): Germany 1-1 Paraguay -> avanza Paraguay
  [REAL] Dieciseisavos de final (75): Netherlands 1-1 Morocco -> avanza Morocco
  [REAL] Dieciseisavos de final (76): Brazil 2-1 Japan -> avanza Brazil
  [REAL] Dieciseisavos de final (77): France 3-0 Sweden -> avanza France
  [REAL] Dieciseisavos de final (78): Ivory Coast 1-2 Norway -> avanza Norway
  [REAL] Dieciseisavos de final (79): Mexico 2-0 Ecuador -> avanza Mexico
  [REAL] Dieciseisavos de final (80): England 2-1 DR Congo -> avanza England
  [REAL] Dieciseisavos de final (81): United States 2-0 Bosnia and Herzegovina -> avanza United States
  [REAL] Dieciseisavos de final (82): Belgium 2-2 Senegal -> avanza Belgium
  [REAL] Dieciseisavos de final (83): Portugal 2-1 Croatia -> avanza Portugal
  [REAL] Dieciseisavos de final (84): Spain 3-0 Austria -> avanza Spain
  [REAL] Dieciseisavos de final (85): Switzerland 2-0 Algeri

## 5.5 Persistencia

In [6]:
ruta_cuadro = DIR_RESULTS / "cuadro_completo.csv"
df_cuadro.to_csv(ruta_cuadro, index=False)
print(f"Guardado: {ruta_cuadro} ({len(df_cuadro)} partidos)")
df_cuadro

Guardado: /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/results/cuadro_completo.csv (32 partidos)


,num_partido,ronda,fecha,equipo_a,equipo_b,es_neutral,marcador,ganador,jugado,prob_gana_a,prob_gana_b
0,73,Dieciseisavos de final,2026-06-28,South Africa,Canada,True,0-1,Canada,True,NaN,NaN
1,74,Dieciseisavos de final,2026-06-29,Germany,Paraguay,True,1-1,Paraguay,True,NaN,NaN
2,75,Dieciseisavos de final,2026-06-29,Netherlands,Morocco,True,1-1,Morocco,True,NaN,NaN
3,76,Dieciseisavos de final,2026-06-29,Brazil,Japan,True,2-1,Brazil,True,NaN,NaN
4,77,Dieciseisavos de final,2026-06-30,France,Sweden,True,3-0,France,True,NaN,NaN
5,78,Dieciseisavos de final,2026-06-30,Ivory Coast,Norway,True,1-2,Norway,True,NaN,NaN
6,79,Dieciseisavos de final,2026-06-30,Mexico,Ecuador,False,2-0,Mexico,True,NaN,NaN
7,80,Dieciseisavos de final,2026-07-01,England,DR Congo,True,2-1,England,True,NaN,NaN
8,81,Dieciseisavos de final,2026-07-01,United States,Bosnia and Herzegovina,False,2-0,United States,True,NaN,NaN
9,82,Dieciseisavos de final,2026-07-01,Belgium,Senegal,True,2-2,Belgium,True,NaN,NaN


## 5.6 Bracket HTML con banderas

Página autocontenida (sin dependencias externas) con una columna por ronda. Cada partido
muestra la bandera y el nombre de cada selección, el marcador (real o previsto) y quién
avanza; los partidos ya jugados y los previstos se distinguen visualmente (color de
`results` del proyecto: verde para lo ya confirmado, azul/neutro para la predicción).

In [7]:
# Emoji de bandera vía secuencia Unicode de indicadores regionales (2 letras
# ISO 3166-1), salvo Inglaterra, que no tiene código de país propio -- usa la
# secuencia de subdivisión (GB-ENG) en vez de recurrir al Reino Unido.
ISO2_POR_EQUIPO = {
    "South Africa": "ZA", "Canada": "CA", "Netherlands": "NL", "Morocco": "MA",
    "Germany": "DE", "Paraguay": "PY", "Brazil": "BR", "Japan": "JP",
    "France": "FR", "Sweden": "SE", "Ivory Coast": "CI", "Norway": "NO",
    "Mexico": "MX", "Ecuador": "EC", "DR Congo": "CD", "United States": "US",
    "Bosnia and Herzegovina": "BA", "Belgium": "BE", "Senegal": "SN",
    "Portugal": "PT", "Croatia": "HR", "Spain": "ES", "Austria": "AT",
    "Switzerland": "CH", "Algeria": "DZ", "Argentina": "AR", "Cape Verde": "CV",
    "Colombia": "CO", "Ghana": "GH", "Australia": "AU", "Egypt": "EG",
}
BANDERA_INGLATERRA = "\U0001F3F4\U000E0067\U000E0062\U000E0065\U000E006E\U000E0067\U000E007F"


def bandera(equipo: str) -> str:
    if equipo == "England":
        return BANDERA_INGLATERRA
    iso2 = ISO2_POR_EQUIPO[equipo]
    return "".join(chr(0x1F1E6 + ord(c) - ord("A")) for c in iso2)


ORDEN_RONDAS = ["Dieciseisavos de final", "Octavos de final", "Cuartos de final",
                "Semifinales", "Tercer puesto", "Final"]


def tarjeta_partido(fila: pd.Series) -> str:
    a_gana = fila["ganador"] == fila["equipo_a"]
    clase_a = "ganador" if a_gana else "perdedor"
    clase_b = "perdedor" if a_gana else "ganador"
    etiqueta = ('<span class="badge badge-real">JUGADO</span>' if fila["jugado"]
                else '<span class="badge badge-previsto">PREVISTO</span>')
    sub = "" if fila["jugado"] else (
        f'<div class="prob">P({fila["equipo_a"]})={fila["prob_gana_a"]:.0%} '
        f'· P({fila["equipo_b"]})={fila["prob_gana_b"]:.0%}</div>')
    goles_a, goles_b = fila["marcador"].split("-")
    return f'''
    <div class="partido">
      <div class="cabecera">{etiqueta}<span class="fecha">{fila["fecha"].strftime("%d %b")}</span></div>
      <div class="equipo {clase_a}"><span class="flag">{bandera(fila["equipo_a"])}</span>
        <span class="nombre">{fila["equipo_a"]}</span><span class="goles">{goles_a}</span></div>
      <div class="equipo {clase_b}"><span class="flag">{bandera(fila["equipo_b"])}</span>
        <span class="nombre">{fila["equipo_b"]}</span><span class="goles">{goles_b}</span></div>
      {sub}
    </div>'''


columnas_html = []
for ronda in ORDEN_RONDAS:
    partidos_ronda = df_cuadro[df_cuadro["ronda"] == ronda].sort_values("num_partido")
    if partidos_ronda.empty:
        continue
    tarjetas = "\n".join(tarjeta_partido(fila) for _, fila in partidos_ronda.iterrows())
    columnas_html.append(f'<section class="ronda ronda-{"".join(ronda.split())}"><h2>{ronda}</h2>{tarjetas}</section>')

campeon = ganadores[104]
html = f'''<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<title>Cuadro de eliminatoria — Mundial 2026</title>
<style>
  :root {{
    --surface-1: #fcfcfb; --page: #f9f9f7; --ink-1: #0b0b0b; --ink-2: #52514e;
    --ink-muted: #898781; --line: #e1e0d9; --good: #0ca30c; --previsto: #2a78d6;
    --winner-bg: #eef6ee;
  }}
  @media (prefers-color-scheme: dark) {{
    :root {{
      --surface-1: #1a1a19; --page: #0d0d0d; --ink-1: #ffffff; --ink-2: #c3c2b7;
      --ink-muted: #898781; --line: #2c2c2a; --good: #0ca30c; --previsto: #3987e5;
      --winner-bg: #14231a;
    }}
  }}
  * {{ box-sizing: border-box; }}
  body {{
    margin: 0; padding: 24px; background: var(--page); color: var(--ink-1);
    font-family: system-ui, -apple-system, "Segoe UI", sans-serif;
  }}
  h1 {{ font-size: 1.4rem; margin: 0 0 4px; }}
  .subtitulo {{ color: var(--ink-2); margin: 0 0 24px; font-size: 0.9rem; }}
  .campeon {{ font-size: 1.1rem; margin: 0 0 24px; padding: 12px 16px; border-radius: 10px;
    background: var(--winner-bg); border: 1px solid var(--line); display: inline-block; }}
  .cuadro {{ display: flex; gap: 16px; overflow-x: auto; padding-bottom: 12px; }}
  .ronda {{ min-width: 240px; flex: 0 0 auto; }}
  .ronda h2 {{ font-size: 0.85rem; text-transform: uppercase; letter-spacing: 0.04em;
    color: var(--ink-muted); border-bottom: 1px solid var(--line); padding-bottom: 6px; }}
  .partido {{ background: var(--surface-1); border: 1px solid var(--line); border-radius: 10px;
    padding: 10px 12px; margin-bottom: 14px; }}
  .cabecera {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px; }}
  .fecha {{ font-size: 0.72rem; color: var(--ink-muted); }}
  .badge {{ font-size: 0.65rem; font-weight: 700; letter-spacing: 0.03em; padding: 2px 6px; border-radius: 4px; }}
  .badge-real {{ color: var(--good); border: 1px solid var(--good); }}
  .badge-previsto {{ color: var(--previsto); border: 1px solid var(--previsto); }}
  .equipo {{ display: flex; align-items: center; gap: 8px; padding: 3px 0; font-size: 0.88rem; }}
  .equipo .flag {{ font-size: 1.1rem; }}
  .equipo .nombre {{ flex: 1; }}
  .equipo .goles {{ font-variant-numeric: tabular-nums; font-weight: 600; }}
  .equipo.ganador {{ font-weight: 700; }}
  .equipo.perdedor {{ color: var(--ink-2); opacity: 0.7; }}
  .prob {{ font-size: 0.68rem; color: var(--ink-muted); margin-top: 4px; }}
</style>
</head>
<body>
  <h1>Cuadro de eliminatoria — Mundial 2026</h1>
  <p class="subtitulo">Modelo: {familia_modelo.upper()} + Dixon-Coles (checkpoint '{etapa}') ·
    verde = ya jugado, azul = previsto por el modelo</p>
  <div class="campeon">🏆 Campeón previsto: <strong>{bandera(campeon)} {campeon}</strong></div>
  <div class="cuadro">
    {"".join(columnas_html)}
  </div>
</body>
</html>'''

ruta_html = DIR_RESULTS / "bracket.html"
ruta_html.write_text(html, encoding="utf-8")
print(f"Guardado: {ruta_html}")

Guardado: /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/results/bracket.html
